In [ ]:
# Practice Problems Day 7

### Author:

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**Reading:** Geron chapter 6, the sections on **Boosting** (AdaBoost and Gradient Boosting) and **Stacking**.

GOALS:

1. Fit a boosted model and compare it to the Day 6 forest on the same tumors.
2. Let XGBoost pick its own number of trees with early stopping, and see what the learning rate does to that number.
3. Use repeated cross validation to decide whether two close models are really different.
4. Find out for yourself whether more trees always overfit.

**How much of this should you write yourself?** Every problem says. Problem 1 gives you the code with blanks. Problems 2 and 3 list the steps. Problem 4 is by hand. Problem 5 just tells you what to do. Same breast cancer data as the Day 5 and Day 6 homework, so the loading and splitting code is code you have written twice.

**The test set stays closed.** You opened it in the Day 5 and Day 6 homework. Everything here happens on validation and cross validation.

**How to turn this in.** Your repository on GitHub **is** your submission. There is no Pull Request to open any more.

Manage your git however you like. Use branches if you want them, or commit straight to `main` if you do not. What I need is only this:

1. The finished work is on your **`main`** branch.
2. It is **pushed to GitHub** before the deadline.
3. Your name is on the **Author** line at the top of this notebook.

```bash
git add .
git commit -m "Day 7 homework"
git push
```

I grade from whatever is on GitHub at the deadline. If it is not pushed, I cannot see it.

This is part of **HW 4, due Sunday 9/27 at 11:59pm**, along with the Day 8 problems and **Weekly Homework 4**.

---

## Problem 1: Boosting on the tumors (fill in the blanks)

**What to copy, what to write:** Fill in the blanks marked `____`. Steps 1 and 2 are the same as the Day 5 and Day 6 homework and are done for you.

**1a.** Fill in the blanks and run it. Report the training accuracy and the validation average precision.

**1b.** In the Day 6 homework, the random forest got a validation average precision of **0.986** on these same tumors. How does your boosted model compare?

**1c.** Look at `max_depth=3` in the code. In the Day 6 forest, every tree was grown all the way down. Why do boosted models use shallow trees instead? (One or two sentences. The Day 7 notes and Geron both talk about this.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, average_precision_score

# Step 1: load the data, 1 means MALIGNANT (same as the Day 5 and Day 6 homework)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series((data.target == 0).astype(int))

# Step 2: test set off first, then a validation set. The test set stays closed today
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42)

# Step 3: a boosted model
boosted = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=____,      # the usual first try from the notes
    max_depth=3,
    random_state=42)
boosted.fit(____, ____)      # the training features and labels. No scaling needed, these are trees

# Step 4: score it
print("training accuracy:", round(accuracy_score(y_train, boosted.predict(X_train)), 3))
y_prob_boost = boosted.predict_proba(____)[:, ____]    # the VALIDATION features, and the column for class 1
print("validation average precision:", round(average_precision_score(y_valid, y_prob_boost), 3))

*Your answers here.*

---

## Problem 2: Let XGBoost stop itself (steps outlined)

**What to copy, what to write:** The steps are comments in the cell. The early stopping cell in the Day 7 notes is the pattern to follow.

**2a.** Fit an XGBoost model with `n_estimators=2000`, `learning_rate=0.05`, `max_depth=3`, `eval_metric="aucpr"`, and `early_stopping_rounds=50`, watching the validation set. Print `best_iteration` and the validation average precision.

**2b.** Do it again with `learning_rate=0.3` and nothing else changed. Print the same two numbers.

**2c.** Compare the two. What happened to the number of trees, and what happened to the score? Explain it with the learning rate section of the notes.

**2d.** Neither model used anything close to 2000 trees. If you had instead set `n_estimators=2000` with no early stopping, what would have happened, and how would you have found out?

In [ ]:
import xgboost as xgb

# Step 1 (2a): an XGBClassifier with the settings above. Note early_stopping_rounds goes in
#              the constructor, and eval_set goes in .fit()


# Step 2 (2a): print best_iteration and the validation average precision


# Step 3 (2b): the same thing with learning_rate=0.3

*Your answers here.*

---

## Problem 3: Are they really different? (steps outlined)

In 1b you compared 0.975 against the forest's 0.986. Now find out whether that gap is real.

**What to copy, what to write:** The `RepeatedStratifiedKFold` cell in the notes is the pattern. Change the models and the data.

**3a.** Use `cross_val_score` with `RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=42)` and `scoring="average_precision"` on **`X_train_full`** (the full training set, not the split you fit on). Do it for a random forest with 200 trees and for your boosted model. Print each one's mean, standard deviation, and worst and best fold.

**3b.** Which model has the higher mean? By how much? How does that gap compare to the standard deviation?

**3c.** Write the one sentence you would put in a report about which model is better.

**3d.** The test set would settle it in one line. Why are you not allowed to use it here?

In [ ]:
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier

# Step 1 (3a): make the cv object


# Step 2 (3a): for each of the two models, cross_val_score on X_train_full and y_train_full,
#              then print the mean, the standard deviation, and the min and max
#              (this fits each model 20 times, so give it a few seconds)

*Your answers here.*

---

## Problem 4: By hand, no code

**4a.** A boosted model currently predicts **3.0** for a house, and the true price is **4.2**. What is the residual the next tree will be trained on? If that next tree predicts **1.0** for this house and the learning rate is **0.1**, what does the model predict for the house after the tree is added?

**4b.** You have a boosted model with `learning_rate=0.2` and 100 trees. You want to try `learning_rate=0.02`. Roughly how many trees should you expect to need, and why?

**4c.** Your friend has a 16 core laptop and says "I will just use all the cores, boosting will be as fast as the forest." What is wrong with that plan?

**4d.** Early stopping needs a set of data to watch. Explain why it must not watch the test set, in terms of what the test set is for.

*Your answers here.*

---

## Problem 5: Does it always overfit? (just the task)

In the notes, adding trees past about 50 made the model **worse** on the fake curve: validation error went from 0.362 up to 0.476 by 2000 trees. Find out whether the same thing happens on the tumors.

**5a.** Fit a gradient boosted model with `max_depth=3`, `learning_rate=0.1`, and 10, 50, 200, 1000, and 3000 trees. For each one, print the validation average precision. (The 3000 tree fit takes a few seconds.)

**5b.** Does the score get worse as trees pile up, the way it did on the fake curve?

**5c.** Explain what you found. Two or three sentences. Think about how much noise there is in each dataset: the fake curve had a standard deviation of 0.35 added to every point, while these tumor measurements come from real images of real cells and the two classes barely overlap.

**5d.** Given 5b, does that mean you can always set `n_estimators` as high as you like and stop worrying? Say what you would actually do on a new dataset.

In [ ]:
# your code here

*Your answers here.*